In [0]:
# Databricks notebook source
# MAGIC 

In [0]:
%run ./shared_audit_utils

In [0]:
# Import libraries
from pyspark.sql.functions import *
from pyspark.sql import DataFrame
import logging
from datetime import datetime

DataFrame[]

In [0]:
# Run id
run_id = get_run_id()
start_time = datetime.now()

In [0]:
# Logging cofiguration
logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s %(levelname)s %(message)s"
)
logger = logging.getLogger("silver_pipeline")

In [0]:
# Configuration
PIPELINE_NAME = "online_retail_silver"

config = (
    spark.table("workspace.default.pipeline_config")
    .filter(
        f"pipeline_name = '{PIPELINE_NAME}'"
    )
    .first()
)

if config is None:
    raise Exception(
        f"Pipeline config not found: {PIPELINE_NAME}"
    )

BRONZE_TABLE = config.source_name
SILVER_TABLE = config.target_table

logger.info(
    f"Source Table: {BRONZE_TABLE}"
)

logger.info(
    f"Target Table: {SILVER_TABLE}"
)

2026-06-06 20:20:05,237 INFO Source Table: workspace.default.bronze_online_retail
2026-06-06 20:20:05,238 INFO Target Table: workspace.default.silver_online_retail


In [0]:
# Read source
def read_bronze() -> DataFrame:
    logger.info(f"Reading bronze table: {BRONZE_TABLE}")
    return (
        spark.table(BRONZE_TABLE)
    )
                

2026-06-06 20:20:05,470 INFO Received command c on object id p0


In [0]:
# Validation
def validate_source(df:DataFrame):
    logger.info("Running source validation")
    if df.limit(1).count() == 0:
        raise Exception(
            f"Source table: {BRONZE_TABLE} is empty"
            )


In [0]:
# Get validation rules 
def get_validation_rules(
        pipeline_name):

    return (
        spark.table(
            "workspace.default.validation_rules"
        )
        .filter(
            f"""
            pipeline_name =
            '{pipeline_name}'
            AND is_active = true
            """
        )
        .collect()
    )

In [0]:
# Apply validation rules
def apply_validation_rules(
        df,
        pipeline_name):

    rules = get_validation_rules(
        pipeline_name
    )

    logger.info(
        f"Found {len(rules)} validation rules"
    )

    for rule in rules:

        logger.info(
            f"Applying rule: "
            f"{rule.rule_name}"
        )

        df = df.filter(
            rule.rule_expression
        )

    return df

In [0]:
# Standardize Date
def convert_invoice_date(df:DataFrame) -> DataFrame:
    logger.info("Converting InvoiceDate")
    return (
        df.withColumn("InvoiceDate", try_to_timestamp(col("InvoiceDate"), lit("dd-MM-yyyy HH:mm")))
    )

In [0]:
# Validate dates
def validate_dates(df: DataFrame):
    invalid_dates = (
        df.filter(col("InvoiceDate").isNull()).count()
    )
    logger.info(f"Invalid dates found: {invalid_dates}")
    if invalid_dates > 0:
        raise Exception(
            f"{invalid_dates} invalid dates detected"
        )

In [0]:
# Business column
def add_sales_amount(df:DataFrame) -> DataFrame:
    logger.info("Adding sales amount")
    return (
        df.withColumn("SalesAmount", round(col("Quantity") * col("UnitPrice"), 2))
    )



In [0]:
# Metadata columns
def add_metadata(df: DataFrame) -> DataFrame:

    return (
        df
        .withColumn("load_timestamp",current_timestamp())
        .withColumn("pipeline_name",lit(PIPELINE_NAME))
    )

In [0]:
# Write to silver
def write_silver(df:DataFrame):
    logger.info(f"Writing silver table: {SILVER_TABLE}")
    return (
        df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
    )


In [0]:
# Audit Metrics
def log_pipeline_metrics(input_count,output_count,df):
    logger.info(f"Input Count: {input_count}")
    logger.info(f"Output Count: {output_count}")
    logger.info(f"Rejected Count: {input_count-output_count}")
    logger.info(
        f"Null CustomerID: "
        f"{df.filter(col('CustomerID').isNull()).count()}"
    )
    logger.info(
        f"Null Description: "
        f"{df.filter(col('Description').isNull()).count()}"
    )

In [0]:
# Main
def main(run_id,start_time):
    logger.info("Starting silver pipeline")
    bronze_df = read_bronze()
    validate_source(bronze_df)
    input_count = bronze_df.count()
    silver_df = (
        bronze_df
        .transform(
            lambda df:
            apply_validation_rules(
                df,
                PIPELINE_NAME
            )
        )
        .transform(convert_invoice_date)
        .transform(add_sales_amount)
        .transform(add_metadata)
        )
    
    output_count = silver_df.count()
    validate_dates(silver_df)
    log_pipeline_metrics(input_count,output_count,silver_df)
    write_silver(silver_df)
    logger.info("Finished silver pipeline")

    end_time = datetime.now()
    # Write audit record
    # logger.info("Writing logs to audit table")
    write_audit_record(
        run_id=run_id,
        pipeline_name=PIPELINE_NAME,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        input_count=input_count,
        output_count=output_count,
        error_message=None
    )

In [0]:
# Entry Point
if __name__ == "__main__":

    run_id = get_run_id()
    start_time = datetime.now()

    try:

        main(
            run_id,
            start_time
        )

    except Exception as e:

        end_time = datetime.now()

        write_audit_record(
            run_id=run_id,
            pipeline_name=PIPELINE_NAME,
            start_time=start_time,
            end_time=end_time,
            status="FAILED",
            input_count=0,
            output_count=0,
            error_message=str(e)
        )

        logger.exception(
            f"Pipeline {PIPELINE_NAME} failed: {e}"
        )

        raise

2026-06-06 20:20:18,543 INFO Starting silver pipeline
2026-06-06 20:20:18,544 INFO Reading bronze table: workspace.default.bronze_online_retail
2026-06-06 20:20:18,544 INFO Running source validation
2026-06-06 20:20:19,991 INFO Found 3 validation rules
2026-06-06 20:20:19,993 INFO Applying rule: positive_quantity
2026-06-06 20:20:19,993 INFO Applying rule: valid_description
2026-06-06 20:20:19,994 INFO Applying rule: not_cancelled
2026-06-06 20:20:19,995 INFO Converting InvoiceDate
2026-06-06 20:20:20,032 INFO Adding sales amount
2026-06-06 20:20:21,718 INFO Invalid dates found: 0
2026-06-06 20:20:21,719 INFO Input Count: 541909
2026-06-06 20:20:21,720 INFO Output Count: 530693
2026-06-06 20:20:21,721 INFO Rejected Count: 11216
2026-06-06 20:20:22,324 INFO Null CustomerID: 132769
2026-06-06 20:20:22,925 INFO Null Description: 0
2026-06-06 20:20:22,926 INFO Writing silver table: workspace.default.silver_online_retail
2026-06-06 20:20:26,333 INFO Finished silver pipeline
2026-06-06 20:20